# 01 - Dataset preparation for nuScenes and subset creation

## Goal
This notebook documents the dataset preparation workflow for nuScenes in MMDetection3D and creates a reproducible reduced subset for budget-constrained experiments.

## Why this is needed
MMDetection3D normally prepares nuScenes by generating metadata `.pkl` files from the dataset root.  
For reduced-budget experiments, a clean approach is to define a subset at the **scene level** and then create reduced info files that contain only those scenes.

## Scientific choice
I use a **scene-level subset** rather than random frame dropping because:
- it preserves temporal and contextual consistency within scenes
- it is easier to reproduce
- it is cleaner for later comparison across models

## Planned output
This notebook should:
1. verify the dataset layout
2. verify that the standard nuScenes info files exist
3. inspect train/val scene coverage
4. create a fixed subset of train scenes
5. filter the existing info files into subset `.pkl` files
6. save the exact subset definition for reproducibility

# Setup and paths

In [2]:
from pathlib import Path
import os
import json
import pickle
from collections import Counter

# =========================
# User-specific paths
# =========================
PROJECT_ROOT = Path("/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning")
MMDET3D_ROOT = PROJECT_ROOT / "external" / "mmdetection3d"
NUSCENES_ROOT = Path("/rs_scratch/users/ae04q066/nuscenes_full")

# Folder where MMDetection3D expects data by default
MMDET3D_DATA_ROOT = MMDET3D_ROOT / "data" / "nuscenes"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("MMDET3D_ROOT     :", MMDET3D_ROOT)
print("NUSCENES_ROOT    :", NUSCENES_ROOT)
print("MMDET3D_DATA_ROOT:", MMDET3D_DATA_ROOT)

print("\nExistence check:")
for p in [PROJECT_ROOT, MMDET3D_ROOT, NUSCENES_ROOT]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

PROJECT_ROOT     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning
MMDET3D_ROOT     : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d
NUSCENES_ROOT    : /rs_scratch/users/ae04q066/nuscenes_full
MMDET3D_DATA_ROOT: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes

Existence check:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning: OK
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d: OK
/rs_scratch/users/ae04q066/nuscenes_full: OK


# Check the raw nuScenes folder structure

In [3]:
expected_entries = [
    "maps",
    "samples",
    "sweeps",
    "v1.0-trainval",
]

print("Checking expected nuScenes entries:\n")
for name in expected_entries:
    p = NUSCENES_ROOT / name
    print(f"{name:<15} -> {'OK' if p.exists() else 'MISSING'}")

Checking expected nuScenes entries:

maps            -> OK
samples         -> OK
sweeps          -> OK
v1.0-trainval   -> OK


# Check whether MMDetection3D already has prepared info files

In [4]:
candidate_info_files = [
    NUSCENES_ROOT / "nuscenes_infos_train.pkl",
    NUSCENES_ROOT / "nuscenes_infos_val.pkl",
    MMDET3D_DATA_ROOT / "nuscenes_infos_train.pkl",
    MMDET3D_DATA_ROOT / "nuscenes_infos_val.pkl",
]

print("Checking candidate info files:\n")
for p in candidate_info_files:
    print(f"{p} -> {'FOUND' if p.exists() else 'NOT FOUND'}")

Checking candidate info files:

/rs_scratch/users/ae04q066/nuscenes_full/nuscenes_infos_train.pkl -> FOUND
/rs_scratch/users/ae04q066/nuscenes_full/nuscenes_infos_val.pkl -> FOUND
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_train.pkl -> FOUND
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_val.pkl -> FOUND


## Notes on standard MMDetection3D preparation

The standard nuScenes preparation step in MMDetection3D creates info files such as:
- `nuscenes_infos_train.pkl`
- `nuscenes_infos_val.pkl`

These files are then referenced by training configs.

In this project, I first check whether those files already exist.  
If they do, I filter them to build a reproducible subset.  
This is simpler and safer than rewriting the whole conversion pipeline.

## Decision rule

There are two possible cases:

### Case A
The standard `nuscenes_infos_train.pkl` and `nuscenes_infos_val.pkl` already exist.

Then I will:
- load them
- inspect their structure
- create a subset by filtering scenes
- save new subset `.pkl` files

### Case B
The info files do not exist yet.

Then I will first generate them using the standard MMDetection3D preparation command and only after that create the subset.

# Selecting the info files for subset creation

In [5]:
from pathlib import Path

# Prefer the files inside the MMDetection3D data folder, since that is the
# location normally referenced by configs. Fall back to the raw dataset folder.
train_info_path = MMDET3D_DATA_ROOT / "nuscenes_infos_train.pkl"
val_info_path = MMDET3D_DATA_ROOT / "nuscenes_infos_val.pkl"

if not train_info_path.exists():
    train_info_path = NUSCENES_ROOT / "nuscenes_infos_train.pkl"

if not val_info_path.exists():
    val_info_path = NUSCENES_ROOT / "nuscenes_infos_val.pkl"

print("Selected train info file:", train_info_path)
print("Selected val info file  :", val_info_path)

Selected train info file: /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_train.pkl
Selected val info file  : /storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/nuscenes_infos_val.pkl


# Loading info files and inspecting structure

In this step, I load the precomputed nuScenes info files and inspect their structure.

This is necessary because:
- MMDetection3D info files can have slightly different formats depending on version
- I need to identify where the list of samples is stored

In [6]:
import pickle

with open(train_info_path, "rb") as f:
    train_data = pickle.load(f)

with open(val_info_path, "rb") as f:
    val_data = pickle.load(f)

print("Top-level type of train_data:", type(train_data))
print("Top-level type of val_data  :", type(val_data))

if isinstance(train_data, dict):
    print("\nTrain keys:", list(train_data.keys()))
if isinstance(val_data, dict):
    print("Val keys  :", list(val_data.keys()))

Top-level type of train_data: <class 'dict'>
Top-level type of val_data  : <class 'dict'>

Train keys: ['metainfo', 'data_list']
Val keys  : ['metainfo', 'data_list']


# Extracting the sample list

The loaded info files are dictionaries with:
- `metainfo`
- `data_list`

The actual per-sample records are stored in `data_list`, so I now extract that list and inspect one sample record to understand which fields are available for subset filtering.

In [7]:
def get_infos_container(data):
    """
    Return the main list of sample infos from a loaded nuScenes info file.
    Supports common MMDetection3D formats.
    """
    if isinstance(data, dict):
        if "infos" in data:
            return data["infos"]
        if "data_list" in data:
            return data["data_list"]
    elif isinstance(data, list):
        return data
    raise ValueError("Unsupported info file structure")

train_infos = get_infos_container(train_data)
val_infos = get_infos_container(val_data)

print("Number of train samples:", len(train_infos))
print("Number of val samples  :", len(val_infos))

print("\nType of one train sample:", type(train_infos[0]))
print("Keys of first train sample:")
print(list(train_infos[0].keys()))

Number of train samples: 28130
Number of val samples  : 6019

Type of one train sample: <class 'dict'>
Keys of first train sample:
['sample_idx', 'token', 'timestamp', 'ego2global', 'images', 'lidar_points', 'instances', 'cam_instances']


# Inspecting one sample record in more detail

The top-level sample record does not directly show a `scene_token`, so I now inspect the nested fields.

This is important because the scene identity may be stored inside:
- `lidar_points`
- `images`
- another nested metadata structure

If scene information is not directly available, I will reconstruct it from the nuScenes metadata tables later.

In [8]:
sample0 = train_infos[0]

for key, value in sample0.items():
    print(f"\n===== {key} =====")
    print("type:", type(value))
    
    if isinstance(value, dict):
        print("dict keys:", list(value.keys()))
    elif isinstance(value, list):
        print("list length:", len(value))
        if len(value) > 0:
            print("type of first element:", type(value[0]))
            if isinstance(value[0], dict):
                print("keys of first element:", list(value[0].keys()))
    else:
        print(value)


===== sample_idx =====
type: <class 'int'>
0

===== token =====
type: <class 'str'>
e93e98b63d3b40209056d129dc53ceee

===== timestamp =====
type: <class 'float'>
1531883530.449377

===== ego2global =====
type: <class 'list'>
list length: 4
type of first element: <class 'list'>

===== images =====
type: <class 'dict'>
dict keys: ['CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_FRONT_LEFT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']

===== lidar_points =====
type: <class 'dict'>
dict keys: ['num_pts_feats', 'lidar_path', 'lidar2ego']

===== instances =====
type: <class 'list'>
list length: 10
type of first element: <class 'dict'>
keys of first element: ['bbox_label', 'bbox_3d', 'bbox_3d_isvalid', 'bbox_label_3d', 'num_lidar_pts', 'num_radar_pts', 'velocity']

===== cam_instances =====
type: <class 'dict'>
dict keys: ['CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_FRONT_LEFT', 'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']


# Checking scene reconstruction through the nuScenes metadata

The prepared MMDetection3D sample records do not directly contain a `scene_token`.

However, each record does contain a nuScenes `token`, which is the sample token.
Using the nuScenes metadata tables, I should be able to map:

sample token -> scene token -> scene name

This would allow reproducible scene-level subset creation even when the scene is not stored explicitly in the prepared info file.

In [9]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(
    version="v1.0-trainval",
    dataroot=str(NUSCENES_ROOT),
    verbose=True,
)

print("\nLoaded nuScenes tables successfully.")
print("Number of scenes :", len(nusc.scene))
print("Number of samples:", len(nusc.sample))

Loading NuScenes tables for version v1.0-trainval...
23 category,
8 attribute,
4 visibility,
64386 instance,
12 sensor,
10200 calibrated_sensor,
2631083 ego_pose,
68 log,
850 scene,
34149 sample,
2631083 sample_data,
1166187 sample_annotation,
4 map,
Done loading in 58.746 seconds.
Reverse indexing ...
Done reverse indexing in 6.9 seconds.

Loaded nuScenes tables successfully.
Number of scenes : 850
Number of samples: 34149


# Verifying sample-to-scene mapping

I now verify that a sample token from the prepared MMDetection3D info file can be matched to:
- a nuScenes sample record
- its parent scene
- the corresponding scene name

If this works, then I can build the subset by selecting a fixed list of scenes and filtering all sample records whose sample token belongs to those scenes.

In [10]:
sample_token_0 = train_infos[0]["token"]

sample_record_0 = nusc.get("sample", sample_token_0)
scene_token_0 = sample_record_0["scene_token"]
scene_record_0 = nusc.get("scene", scene_token_0)

print("Sample token :", sample_token_0)
print("Scene token  :", scene_token_0)
print("Scene name   :", scene_record_0["name"])
print("Description  :", scene_record_0["description"])
print("Log token    :", scene_record_0["log_token"])

Sample token : e93e98b63d3b40209056d129dc53ceee
Scene token  : 73030fb67d3c46cfb5e590168088ae39
Scene name   : scene-0001
Description  : Construction, maneuver between several trucks
Log token    : 6b6513e6c8384cec88775cae30b78c0e


# Building lookup tables for scenes

Since the mapping from sample token to scene works, I now build fast lookup dictionaries for:

- sample token -> scene token
- scene token -> scene name
- scene token -> scene metadata

These lookup tables will make it easy to inspect train/validation scene coverage and later create a reproducible subset.

In [11]:
sample_to_scene = {}
scene_token_to_name = {}
scene_token_to_record = {}

for scene in nusc.scene:
    scene_token_to_name[scene["token"]] = scene["name"]
    scene_token_to_record[scene["token"]] = scene

for sample in nusc.sample:
    sample_to_scene[sample["token"]] = sample["scene_token"]

print("Lookup table sizes:")
print("sample_to_scene      :", len(sample_to_scene))
print("scene_token_to_name  :", len(scene_token_to_name))
print("scene_token_to_record:", len(scene_token_to_record))

Lookup table sizes:
sample_to_scene      : 34149
scene_token_to_name  : 850
scene_token_to_record: 850


# Counting scene coverage in the prepared train and validation infos

I now map every prepared sample record back to its parent scene.

This allows me to check:
- how many distinct scenes are represented in the train info file
- how many distinct scenes are represented in the validation info file
- whether the prepared MMDetection3D infos are consistent with scene-level splitting

In [12]:
train_scene_tokens = sorted({sample_to_scene[x["token"]] for x in train_infos})
val_scene_tokens = sorted({sample_to_scene[x["token"]] for x in val_infos})

train_scene_names = [scene_token_to_name[t] for t in train_scene_tokens]
val_scene_names = [scene_token_to_name[t] for t in val_scene_tokens]

print("Distinct train scenes:", len(train_scene_tokens))
print("Distinct val scenes  :", len(val_scene_tokens))

print("\nFirst 10 train scenes:")
print(train_scene_names[:10])

print("\nFirst 10 val scenes:")
print(val_scene_names[:10])

Distinct train scenes: 700
Distinct val scenes  : 150

First 10 train scenes:
['scene-0234', 'scene-0126', 'scene-0471', 'scene-0854', 'scene-0187', 'scene-0075', 'scene-0139', 'scene-0587', 'scene-0958', 'scene-0515']

First 10 val scenes:
['scene-0101', 'scene-0907', 'scene-1059', 'scene-0922', 'scene-0800', 'scene-0272', 'scene-0625', 'scene-1071', 'scene-0559', 'scene-0921']


# Split consistency result

The prepared MMDetection3D info files map cleanly back to nuScenes scenes.

Observed coverage:
- training info file: 700 distinct scenes
- validation info file: 150 distinct scenes

This matches the expected train/validation partitioning of nuScenes `v1.0-trainval` and confirms that scene-level subset creation is feasible from the existing prepared info files.

# Defining the subset strategy

To reduce training cost while keeping the experiment scientifically clean, I define the subset at the **scene level**.

### Rationale
A scene-level subset is preferred over random sample dropping because it:
- preserves temporal consistency within each driving scene
- keeps the subset easy to reproduce
- allows fair comparison across models if the same scene list is reused

### Planned strategy
For the first reduced-budget experiment, I will:
- keep the validation set unchanged
- select a fixed fraction of the **training scenes**
- include **all samples** belonging to the selected training scenes

This gives a smaller training set while preserving a stable validation benchmark.

# Choosing the subset size

For the first reduced-budget run, I create a fixed subset of the training scenes.

I use a deterministic selection rule so that:
- the subset can be reproduced later
- the same subset can be reused across different models
- comparisons remain fair

In this notebook, I start with a target fraction of the training scenes and then convert it into an exact number of scenes.

In [13]:
# =========================
# Subset size configuration
# =========================
subset_fraction = 0.20   # change later if needed, e.g. 0.10, 0.20, 0.25

num_train_scenes_total = len(train_scene_tokens)
num_train_scenes_subset = max(1, round(num_train_scenes_total * subset_fraction))

print("Total train scenes      :", num_train_scenes_total)
print("Requested fraction      :", subset_fraction)
print("Selected number of scenes:", num_train_scenes_subset)

Total train scenes      : 700
Requested fraction      : 0.2
Selected number of scenes: 140


# Creating a deterministic training-scene subset

I now select a fixed subset of the training scenes.

To make the selection reproducible, I sort the training scene names and take the first `N` scenes.
This is simple, deterministic, and easy to document in the report.

Later, the exact selected scene names will be saved to disk so the subset can be reused across experiments.

In [14]:
train_scene_name_to_token = {
    scene_token_to_name[t]: t for t in train_scene_tokens
}

sorted_train_scene_names = sorted(train_scene_name_to_token.keys())
selected_train_scene_names = sorted_train_scene_names[:num_train_scenes_subset]
selected_train_scene_tokens = {
    train_scene_name_to_token[name] for name in selected_train_scene_names
}

print("Number of selected train scenes:", len(selected_train_scene_names))

print("\nFirst 20 selected train scenes:")
print(selected_train_scene_names[:20])

Number of selected train scenes: 140

First 20 selected train scenes:
['scene-0001', 'scene-0002', 'scene-0004', 'scene-0005', 'scene-0006', 'scene-0007', 'scene-0008', 'scene-0009', 'scene-0010', 'scene-0011', 'scene-0019', 'scene-0020', 'scene-0021', 'scene-0022', 'scene-0023', 'scene-0024', 'scene-0025', 'scene-0026', 'scene-0027', 'scene-0028']


# Saving the subset definition

The exact list of selected training scenes is saved to disk.

This is important for reproducibility because:
- the same subset can be reused for all models
- the experiment setup can be documented precisely
- later reruns do not depend on notebook state

In [15]:
subset_dir = MMDET3D_DATA_ROOT / "subsets"
subset_dir.mkdir(parents=True, exist_ok=True)

subset_name = f"train_scene_subset_{int(subset_fraction * 100):02d}pct"
subset_json_path = subset_dir / f"{subset_name}.json"

subset_definition = {
    "subset_name": subset_name,
    "subset_fraction": subset_fraction,
    "num_train_scenes_total": num_train_scenes_total,
    "num_train_scenes_subset": num_train_scenes_subset,
    "selected_train_scene_names": selected_train_scene_names,
}

with open(subset_json_path, "w") as f:
    json.dump(subset_definition, f, indent=2)

print("Saved subset definition to:")
print(subset_json_path)

Saved subset definition to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/subsets/train_scene_subset_20pct.json


# Filtering the training info file

I now create a reduced training info file by keeping only those samples whose parent scene belongs to the selected training-scene subset.

The validation info file will remain unchanged for now, because keeping validation fixed gives a stable benchmark for comparison.

In [16]:
subset_train_infos = [
    x for x in train_infos
    if sample_to_scene[x["token"]] in selected_train_scene_tokens
]

print("Original number of train samples:", len(train_infos))
print("Subset number of train samples  :", len(subset_train_infos))
print("Retained fraction of samples    :", len(subset_train_infos) / len(train_infos))

Original number of train samples: 28130
Subset number of train samples  : 5552
Retained fraction of samples    : 0.19736935655883397


## Verifying the filtered training subset

Before saving the reduced info file, I verify that:
- only the selected training scenes are present
- the number of distinct scenes matches the intended subset size

This is a simple consistency check to ensure the filtering step worked correctly.

In [17]:
subset_train_scene_tokens_check = sorted({
    sample_to_scene[x["token"]] for x in subset_train_infos
})
subset_train_scene_names_check = [
    scene_token_to_name[t] for t in subset_train_scene_tokens_check
]

print("Distinct scenes in filtered subset:", len(subset_train_scene_tokens_check))
print("Expected number of selected scenes:", len(selected_train_scene_tokens))

print("\nFirst 20 scenes in filtered subset:")
print(subset_train_scene_names_check[:20])

print("\nSubset matches intended scene selection:",
      set(subset_train_scene_tokens_check) == set(selected_train_scene_tokens))

Distinct scenes in filtered subset: 140
Expected number of selected scenes: 140

First 20 scenes in filtered subset:
['scene-0126', 'scene-0187', 'scene-0075', 'scene-0139', 'scene-0021', 'scene-0194', 'scene-0049', 'scene-0025', 'scene-0161', 'scene-0051', 'scene-0154', 'scene-0166', 'scene-0208', 'scene-0027', 'scene-0212', 'scene-0192', 'scene-0149', 'scene-0207', 'scene-0179', 'scene-0004']

Subset matches intended scene selection: True


## Saving the reduced training info file

The reduced training info file is saved in the same MMDetection3D data area.

I preserve the original top-level structure of the info file and only replace the sample list with the filtered subset.
This keeps the file compatible with the MMDetection3D training pipeline.

In [18]:
subset_train_data = dict(train_data)
subset_train_data["data_list"] = subset_train_infos

subset_info_dir = MMDET3D_DATA_ROOT / "subsets"
subset_info_dir.mkdir(parents=True, exist_ok=True)

subset_train_info_path = subset_info_dir / f"nuscenes_infos_train_{int(subset_fraction * 100):02d}pct.pkl"

with open(subset_train_info_path, "wb") as f:
    pickle.dump(subset_train_data, f)

print("Saved reduced training info file to:")
print(subset_train_info_path)
print("\nFile exists:", subset_train_info_path.exists())

Saved reduced training info file to:
/storage/homefs/ae04q066/projects/nuscenes-multimodal-learning/external/mmdetection3d/data/nuscenes/subsets/nuscenes_infos_train_20pct.pkl

File exists: True


# Final result of dataset preparation

A reduced training dataset was successfully created using a scene-level subset.

## Summary
- original training scenes: 700
- selected training scenes: 140 (20%)
- original training samples: 28130
- subset training samples: 5552 (~20%)

## Key design choice
The subset was defined at the **scene level**, ensuring:
- temporal consistency within scenes
- reproducibility via a saved scene list
- fair comparison across models using the same subset

## Files generated
- subset definition:
  `subsets/train_scene_subset_20pct.json`
- reduced training info file:
  `subsets/nuscenes_infos_train_20pct.pkl`

## Validation strategy
The validation set was kept unchanged to maintain a stable benchmark.

# How this subset will be used in training

To train models on this reduced dataset in MMDetection3D:

- set `ann_file` of the training dataset to:
  `data/nuscenes/subsets/nuscenes_infos_train_20pct.pkl`

- keep validation unchanged:
  `data/nuscenes/nuscenes_infos_val.pkl`

This ensures:
- faster training
- consistent evaluation
- direct comparability across models

In [19]:
# Quick sanity check: load subset file again

with open(subset_train_info_path, "rb") as f:
    check_data = pickle.load(f)

check_infos = get_infos_container(check_data)

print("Loaded subset samples:", len(check_infos))
print("Expected samples     :", len(subset_train_infos))
print("Match:", len(check_infos) == len(subset_train_infos))

Loaded subset samples: 5552
Expected samples     : 5552
Match: True
